# Control variables: World Bank contextual indicators

**Owner:** Efrén · **Project:** Internet access and school completion in developing countries (PPOL 5203)

This notebook downloads the country-year **control / contextual variables** from the World Bank API and saves:

| File | Contents |
|---|---|
| `data/control_variables.csv` | Panel: one row per country per year (217 economies, 2000–2025) |
| `data/control_variables_codebook.csv` | What each column means, its source, why we use it, and its coverage |

**To reproduce:** *Run All*. Needs internet access plus `pandas` and `requests`. No API key required.

All economies are kept (including high income) so we can compare; filter to developing countries in the analysis with `income_group`.

In [6]:
import time
from pathlib import Path

import pandas as pd
import requests

#We are going to use the worldbank api via the requests library
API = "https://api.worldbank.org/v2"

# Parameters for the query
START_YEAR = 2000
END_YEAR = 2025

# Works whether Jupyter starts in the repo root or in ingestion/
ROOT = Path.cwd().parent if Path.cwd().name == "ingestion" else Path.cwd()
DATA_DIR = ROOT / "data"

# Traditional 8-country South Asia (see note in section 2)
SOUTH_ASIA = {"AFG", "BGD", "BTN", "IND", "LKA", "MDV", "NPL", "PAK"}

pd.set_option("display.max_columns", 50)

## 1. Which variables

Our question: *is internet use associated with school completion, and does it differ by type of country?*
Each control should either:

- **Confound** the relationship, i.e. drive both internet access and schooling (income, electricity, urbanization, state capacity), or
- **Segment** it, i.e. let us ask whether the effect is stronger in richer, younger, more urban, or better-governed countries.

**Excluded after checking coverage:** poverty rate and Gini (~30% of country-years), adult literacy (~16%), and primary pupil-teacher ratio (stops in 2019; may come from the UNESCO data instead).
Internet and mobile indicators are left to the internet-access ingestion (Nick).

In [7]:
# column name -> (World Bank code, API source, rationale)
# source 2 = World Development Indicators (WDI), 3 = Worldwide Governance Indicators (WGI)
INDICATORS = {
    # Economy: richer countries have both more internet and more schooling
    # GDP per capita PPP (constant 2021 international $)
    "gdp_pc_ppp": ("NY.GDP.PCAP.PP.KD", 2, "Main income control; also used to segment rich vs poor"),
    # GDP per capita (constant 2015 US$)
    "gdp_pc_usd": ("NY.GDP.PCAP.KD", 2, "Backup income measure with wider coverage than PPP"),
    # GDP growth (annual %)
    "gdp_growth": ("NY.GDP.MKTP.KD.ZG", 2, "Economic momentum / shocks (e.g. 2020)"),
    # Employment in agriculture (% of total employment)
    "agri_employment_share": ("SL.AGR.EMPL.ZS", 2, "Economic structure; child labor demand in farming"),
    # Personal remittances, received (% of GDP)
    "remittances_gdp": ("BX.TRF.PWKR.DT.GD.ZS", 2, "Household income from abroad, funds schooling"),
    # Infrastructure: electricity is a precondition for internet AND for studying
    # Access to electricity (% of population)
    "electricity_access": ("EG.ELC.ACCS.ZS", 2, "Key confounder: needed for internet and schooling"),
    # Access to electricity, rural (% of rural population)
    "electricity_access_rural": ("EG.ELC.ACCS.RU.ZS", 2, "Rural infrastructure gap"),
    # Demographics: segment by 'younger' countries, urbanization
    # Population, total
    "population": ("SP.POP.TOTL", 2, "Size; weights for regional averages"),
    # Population ages 0-14 (% of total population)
    "pop_0_14_share": ("SP.POP.0014.TO.ZS", 2, "How young the country is (school-age pressure)"),
    # Population growth (annual %)
    "pop_growth": ("SP.POP.GROW", 2, "Pressure on school capacity"),
    # Urban population (% of total population)
    "urban_share": ("SP.URB.TOTL.IN.ZS", 2, "Cities get internet and schools first"),
    # Population density (people per sq. km of land area)
    "pop_density": ("EN.POP.DNST", 2, "Cost of rolling out networks and schools"),
    # fertility rate, total (births per woman)
    "fertility_rate": ("SP.DYN.TFRT.IN", 2, "Demographic transition stage"),
    # adolescent rate, total (births per 1,000 women ages 15-19)
    "adolescent_fertility": ("SP.ADO.TFRT", 2, "Teen births drive girls' dropout"),
    # Infant mortality (under-5, per 1,000 live births)
    "under5_mortality": ("SH.DYN.MORT", 2, "Broad human-development / health proxy"),
    # Life expectancy at birth, total (years)
    "life_expectancy": ("SP.DYN.LE00.IN", 2, "Broad human-development proxy"),
    # Education system inputs
    # Government expenditure on education (% of GDP)
    "edu_spend_gdp": ("SE.XPD.TOTL.GD.ZS", 2, "Public investment in education (% GDP)"),
    # Government expenditure on education (% of gov. expenditure)
    "edu_spend_govt": ("SE.XPD.TOTL.GB.ZS", 2, "Education priority in the budget (% of spending)"),
    # Compulsory education, duration (years)"
    "compulsory_edu_years": ("SE.COM.DURS", 2, "Legal schooling requirement"),
    # Labor market: opportunity cost of staying in school
    # Unemployment, youth total (% of total labor force ages 15-24) (modeled ILO estimate)
    "youth_unemployment": ("SL.UEM.1524.ZS", 2, "Low job prospects lower the cost of staying in school"),
    # Labor force participation rate, female (% of female population ages 15+) (modeled ILO estimate)
    "female_lfp": ("SL.TLF.CACT.FE.ZS", 2, "Female labor force participation (also useful for Q2)"),
    # Governance / stability
    # Government Effectiveness - Governance estimate (approx. -2.5 to +2.5)
    "wgi_gov_effectiveness": ("GOV_WGI_GE.EST", 3, "State capacity to deliver schools and networks"),
    # Political Stability - Governance estimate (approx. -2.5 to +2.5)
    "wgi_political_stability": ("GOV_WGI_PV.EST", 3, "Conflict / instability proxy"),
    # Rule of Law - Governance estimate (approx. -2.5 to +2.5)
    "wgi_rule_of_law": ("GOV_WGI_RL.EST", 3, "Institutional quality"),
    # Control of Corruption - Governance estimate (approx. -2.5 to +2.5)
    "wgi_control_corruption": ("GOV_WGI_CC.EST", 3, "Institutional quality"),
}
len(INDICATORS)

25

## 2. API helpers and country metadata

In [8]:
def get_json(url, params, retries=4):
    """GET with simple retries; the World Bank API occasionally times out."""
    for attempt in range(retries):
        try:
            # we are going to do the request from "https://api.worldbank.org/v2" using the coresponding params
            resp = requests.get(url, params=params, timeout=90)
            # 500 from the server? -> raise HTTPError
            resp.raise_for_status()
            # return the json with the data
            return resp.json()
        except (requests.RequestException, ValueError):
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def fetch_indicator(code, source):
    """Long table (country_id, iso3_raw, year, value) for one indicator, all countries."""
    params = {"format": "json", "per_page": 20000, "date": f"{START_YEAR}:{END_YEAR}",
              "source": source, "page": 1}
    records = []
    label = code
    while True:
        meta, rows = get_json(f"{API}/country/all/indicator/{code}", params)
        # format the rows for each country
        for r in rows or []:
            label = r["indicator"]["value"]
            records.append({
                # WDI uses ISO2 in country.id; WGI uses ISO3. Keep both to match either.
                "country_id": r["country"]["id"],
                "iso3_raw": r["countryiso3code"],
                "year": int(r["date"]),
                "value": r["value"],
            })
        if params["page"] >= meta["pages"]:
            break
        params["page"] += 1
    return pd.DataFrame(records), label

Country metadata gives region, income group and lending type. Regional aggregates ("World", "South Asia", …) are dropped.

**South Asia note:** the World Bank's current regions put Afghanistan and Pakistan in *"Middle East, North Africa, Afghanistan & Pakistan"*. Since South Asia is central to our question, we add a `south_asia` flag for the traditional 8 countries. **Use `south_asia`, not `region`, for South Asia comparisons.**

In [9]:
# download regions
_, rows = get_json(f"{API}/country", {"format": "json", "per_page": 500})
countries = pd.DataFrame(
    {
        "iso3": c["id"],
        "iso2": c["iso2Code"],
        "country": c["name"],
        "region": c["region"]["value"].strip(),
        "income_group": c["incomeLevel"]["value"].strip(),
        "lending_type": c["lendingType"]["value"].strip(),
    }
    for c in rows
    if c["region"]["value"] != "Aggregates"
)
countries["south_asia"] = countries.iso3.isin(SOUTH_ASIA)

print(len(countries), "countries")
pd.crosstab(countries.region, countries.income_group, margins=True)

217 countries


income_group,High income,Low income,Lower middle income,Upper middle income,All
region,,,,,
East Asia & Pacific,15,1,8,13,37
Europe & Central Asia,40,0,3,15,58
Latin America & Caribbean,19,0,5,18,42
"Middle East, North Africa, Afghanistan & Pakistan",8,3,7,5,23
North America,3,0,0,0,3
South Asia,0,0,4,2,6
Sub-Saharan Africa,1,21,20,6,48
All,86,25,47,59,217


## 3. Download every indicator and build the panel

We start from a full country × year skeleton so missing data shows up as explicit `NaN` rather than missing rows. Takes about 1–2 minutes.

In [10]:
iso2_to_iso3 = dict(zip(countries.iso2, countries.iso3))
valid_iso3 = set(countries.iso3)

panel = countries.merge(pd.DataFrame({"year": range(START_YEAR, END_YEAR + 1)}), how="cross")

codebook = []
# iterate metric by metric
for col, (code, source, why) in INDICATORS.items():
    print(f"Downloading {col:26s} ({code})")
    df, label = fetch_indicator(code, source)
    df["iso3"] = df.country_id.map(iso2_to_iso3).fillna(df.iso3_raw)
    df.loc[df.country_id.isin(valid_iso3), "iso3"] = df.country_id
    df = df[df.iso3.isin(valid_iso3)][["iso3", "year", "value"]]
    df = df.drop_duplicates(["iso3", "year"]).rename(columns={"value": col})
    panel = panel.merge(df, on=["iso3", "year"], how="left")
    # create codebook
    codebook.append({
        "column": col,
        "wb_code": code,
        "source": "WDI" if source == 2 else "WGI",
        "description": label,
        "rationale": why,
        "pct_nonmissing": round(panel[col].notna().mean() * 100, 1),
        "latest_year": panel.loc[panel[col].notna(), "year"].max(),
    })

panel = panel.drop(columns="iso2").sort_values(["iso3", "year"]).reset_index(drop=True)
codebook = pd.DataFrame(codebook)
panel.shape

(5642, 32)

## 4. Save

In [11]:
# save data to csv
DATA_DIR.mkdir(exist_ok=True)
panel.to_csv(DATA_DIR / "control_variables.csv", index=False)
codebook.to_csv(DATA_DIR / "control_variables_codebook.csv", index=False)
print(f"Saved {len(panel):,} rows ({panel.iso3.nunique()} countries x {panel.year.nunique()} years) to {DATA_DIR}")

Saved 5,642 rows (217 countries x 26 years) to /Users/efrenzagal/Documents/GitHub/PPOL-5200-Final-Project---Fall-2026/data


## 5. Sanity checks

**Coverage.** Share of country-years with data: all countries vs. low/middle income only, 2010–2024.

In [12]:
lmic = panel[panel.income_group.isin(["Low income", "Lower middle income", "Upper middle income"])
             & panel.year.between(2010, 2024)]
coverage = codebook[["column", "source", "pct_nonmissing", "latest_year"]].assign(
    pct_nonmissing_lmic_2010_24=lambda d: [round(lmic[c].notna().mean() * 100, 1) for c in d.column]
)
coverage.sort_values("pct_nonmissing_lmic_2010_24")

,column,source,pct_nonmissing,latest_year,pct_nonmissing_lmic_2010_24
16,edu_spend_gdp,WDI,59.7,2025,72.2
17,edu_spend_govt,WDI,59.6,2025,72.4
4,remittances_gdp,WDI,78.4,2025,87.2
18,compulsory_edu_years,WDI,86.3,2025,90.3
6,electricity_access_rural,WDI,89.8,2024,92.7
3,agri_employment_share,WDI,85.9,2025,94.2
20,female_lfp,WDI,85.9,2025,94.2
19,youth_unemployment,WDI,85.9,2025,94.2
0,gdp_pc_ppp,WDI,90.4,2025,95.2
2,gdp_growth,WDI,94.9,2025,97.5


**South Asia flag** should list 8 countries:

In [13]:
panel.loc[panel.south_asia, ["iso3", "country", "region", "income_group"]].drop_duplicates()

,iso3,country,region,income_group
26,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
442,BGD,Bangladesh,South Asia,Lower middle income
754,BTN,Bhutan,South Asia,Lower middle income
2314,IND,India,South Asia,Lower middle income
2938,LKA,Sri Lanka,South Asia,Upper middle income
3224,MDV,Maldives,South Asia,Upper middle income
3796,NPL,Nepal,South Asia,Lower middle income
3900,PAK,Pakistan,"Middle East, North Africa, Afghanistan & Pakistan",Lower middle income


## 6. Notes for the analysis

1. **`income_group` is today's classification**, applied to every year (e.g. Bangladesh is "Lower middle income" even in 2000). For segmenting by wealth over time, prefer GDP-per-capita terciles, or merge the World Bank historical income classifications.
2. **Education spending is ~60% complete.** As a control it will shrink the sample, so treat it as a robustness check rather than in the main model.
3. **Collinearity:** GDP, under-5 mortality, fertility and life expectancy move together. Pick one or two per group in a regression rather than all of them.
4. **Merge key:** join with the internet and UNESCO data on `iso3` + `year`.